---
---

# 📚 Clase 2 — OOP y Tipos de Datos


**ICCE 1032 Estructura De Datos**

**Astrid E. San Martín J.**

**Universidad Mayor**

**Escuela de Ingeniería**

**astrid.sanmartin@umayor.cl**

**Fecha:** 30-04-2026

**Fecha Entrega:** 04-05-2026 23:59hrs.

---
---

# 🧪 Laboratorio: OOP y Tipos de Datos — Caso Estación de Sensores Minerogeofísicos


**Importante:** Este cuaderno **no contiene código**. Está diseñado como una guía paso a paso para que **construyas tu propia solución desde cero**, manteniendo la misma estructura, **sin usar decoradores**, y con foco en el diseño OOP y la correcta elección de tipos de datos.



## ✍️ Cómo usar esta guía
- Avanza **sección por sección**. Cada sección describe **qué debes construir**, **criterios de aceptación**, y **pistas**.
- Mantén un **registro breve** de decisiones de diseño (por qué elegiste ciertos tipos de datos, invariantes, etc.).
- Recuerda: **no se permiten decoradores** en esta versión (evita `@property`, `@dataclass`, etc.).



## ✍️  Contexto del problema
Una empresa de exploración minera opera **estaciones** con **sensores** (p. ej., Resistividad DC e IP/Chargeability) para monitorear zonas de interés. Cada sensor produce **lecturas** con ubicación 3D y metadatos. Necesitan un **modelo de objetos** para:
- Registrar lecturas de manera consistente (unidades, calidad).
- Obtener estadísticas (p. ej., promedio por sensor con umbral de calidad).
- Manejar múltiples sensores por estación.
- Exportar resultados a JSON/CSV para informes.



## 🎯 Objetivos de aprendizaje
- Diseñar un **modelo OOP** con clases, atributos y métodos que representen el dominio.
- Aplicar **tipos de datos** adecuados (numéricos, texto, booleanos, contenedores).
- Establecer **invariantes** y **validaciones** (precondiciones y postcondiciones simples).
- Implementar **herencia** y **polimorfismo** entre sensores.
- Producir **resúmenes** y **exportaciones** (JSON/CSV) desde un diseño limpio.
- Evaluar el diseño con **pruebas** y una breve **medición** comparativa.



## 0️⃣ 0) Empezando: Tipos de datos (conceptual)
**Qué construir:** Un bloque de ejemplo donde se declare variables representativas del dominio (entero, flotante, cadena, booleano, lista, diccionario, conjunto). Eligir nombres y valores significativos.

**Criterios de aceptación:**

- Identificar correctamente al menos **7 tipos/estructuras**: `int`, `float`, `str`, `bool`, `list`, `dict`, `set`.

- Anotar **por qué** se elije cada estructura (ej.: `set` para etiquetas sin repetición).

**Pistas:**

- Considerar `lecturas` como lista; `sensores` como diccionario `id -> Sensor`.

- Definir etiquetas como conjunto para evitar duplicados (p. ej., `{"RES", "IP"}`).



In [1]:
import datetime

# Tipo: int
# Uso: Representa el número de identificación de un sensor o estación si es un entero, o el total de lecturas.
num_sensores = 3

# Tipo: float
# Uso: Coordenadas (x, y, z), valores de lectura, calidad de la lectura.
coordenada_x = 123.45
valor_resistencia = 500.75
calidad_lectura = 0.85

# Tipo: str
# Uso: Nombres de estaciones, IDs de sensores, unidades de medida, timestamps ISO.
nombre_estacion = "Estacion_A"
id_sensor = "RES-001"
unidad_medida = "Ohm·m"
timestamp_iso = "2026-04-30T10:30:00"

# Tipo: bool
# Uso: Indicar si un sensor está activo, si una lectura es válida.
sensor_activo = True
lectura_valida = False

# Tipo: list
# Uso: Colección ordenada de lecturas para un sensor.
lecturas_sensor_ejemplo = [
    {'valor': 450.2, 'unidad': 'Ohm·m', 'calidad': 0.9},
    {'valor': 510.5, 'unidad': 'Ohm·m', 'calidad': 0.8},
    {'valor': 490.0, 'unidad': 'Ohm·m', 'calidad': 0.95}
]

# Tipo: dict
# Uso: Almacenar sensores en una estación (clave: id_sensor, valor: objeto Sensor), o los detalles de una lectura.
datos_lectura = {
    "timestamp": timestamp_iso,
    "x": coordenada_x,
    "y": 200.1,
    "z": 50.0,
    "valor": valor_resistencia,
    "unidad": unidad_medida,
    "calidad": calidad_lectura
}

# Tipo: set
# Uso: Almacenar tipos de sensores o etiquetas únicas para evitar duplicados.
tipos_sensores_disponibles = {"Resistividad DC", "IP/Chargeability", "GPS"}



## 1️⃣ 1) Modelado del dominio (diseño antes de codificar)
**Qué construir:** Un **mini-diagrama textual** y una descripción breve de cada clase/relación.

**Entidades sugeridas:**

- `Lectura`: `timestamp (ISO)`, `x,y,z (float, m)`, `valor (float)`, `unidad (str)`, `calidad (float 0..1)`

- `Sensor` (base): `id (str)`, `tipo (str)`, `unidad_base (str)`, `lecturas (list)`

- Subclases: `ResistividadSensor`, `IPSensor`

- `Estacion`: `nombre (str)`, `ubicacion (lat, lon)` como `tuple(float, float)`, `sensores (dict)`

**Criterios de aceptación:**

- El diagrama refleja **composición** (Estación contiene Sensores; Sensor contiene Lecturas).

- Se listan **invariantes** clave (p. ej., calidad en [0,1], unidad no vacía, timestamp con 'T').

**Pistas:**

- Pensar en **operaciones clave** desde el uso: registrar lecturas, calcular promedio, filtrar por tiempo/calidad.



### Mini-diagrama textual

```mermaid
classDiagram
    class Estacion {
        +nombre: str
        +ubicacion: tuple(float, float)
        +sensores: dict
        +registrar_sensor(sensor: Sensor)
        +obtener_sensor(id: str): Sensor
        +resumen(): dict
    }

    class Sensor {
        +id: str
        +tipo: str
        +unidad_base: str
        +lecturas: list<Lectura>
        +agregar_lectura(lectura: Lectura)
        +contar_lecturas(): int
        +obtener_ultima_lectura(): Lectura
        +calcular_promedio(umbral_calidad: float): float
        +lecturas_en_rango(start_ts: str, end_ts: str): list<Lectura>
    }

    class ResistividadSensor {
        +id: str
        +tipo: str = "Resistividad DC"
        +unidad_base: str
        +lecturas: list<Lectura>
        +aceptar_unidad(unidad: str): bool
    }

    class IPSensor {
        +id: str
        +tipo: str = "IP/Chargeability"
        +unidad_base: str
        +lecturas: list<Lectura>
        +aceptar_unidad(unidad: str): bool
        +calcular_indicador_calidad(): float
    }

    class Lectura {
        +timestamp: str (ISO 8601)
        +x, y, z: float (metros)
        +valor: float
        +unidad: str
        +calidad: float (0.0 a 1.0)
    }

    Estacion "1" *-- "N" Sensor : contiene
    Sensor "1" *-- "N" Lectura : registra
    Sensor <|-- ResistividadSensor : hereda
    Sensor <|-- IPSensor : hereda
```

---

### Descripción de Clases y Relaciones:

1.  **`Lectura`**
    *   **Rol:** Representa una medición individual realizada por un sensor.
    *   **Atributos:**
        *   `timestamp` (str): Fecha y hora de la lectura en formato ISO 8601 (`YYYY-MM-DDTHH:MM:SS`).
        *   `x`, `y`, `z` (float): Coordenadas espaciales (en metros) de donde se tomó la lectura.
        *   `valor` (float): El valor medido por el sensor.
        *   `unidad` (str): La unidad de medida del valor (ej. "Ohm·m", "mV/V").
        *   `calidad` (float): Un indicador de calidad de la lectura, un valor entre 0.0 y 1.0.
    *   **Invariantes:**
        *   `timestamp` debe tener el formato ISO 8601 (`YYYY-MM-DDTHH:MM:SS`).
        *   `x`, `y`, `z`, `valor` deben ser numéricos (float o int).
        *   `unidad` no puede ser una cadena vacía.
        *   `calidad` debe estar en el rango [0.0, 1.0].

2.  **`Sensor`
    *   **Rol:** Define la interfaz y atributos comunes para todos los tipos de sensores.
    *   **Atributos:**
        *   `id` (str): Identificador único del sensor (ej. "RES-001").
        *   `tipo` (str): Tipo general del sensor (ej. "Resistividad DC").
        *   `unidad_base` (str): La unidad de medida principal que este sensor registra.
        *   `lecturas` (list): Una lista interna que almacena objetos `Lectura`.
    *   **Invariantes:**
        *   `id` y `tipo` no pueden ser cadenas vacías.
        *   `unidad_base` no puede ser una cadena vacía.
        *   Solo se pueden agregar objetos de tipo `Lectura` a la lista `lecturas`.
    *   **Operaciones Clave:** `agregar_lectura`, `contar_lecturas`, `obtener_ultima_lectura`, `calcular_promedio`, `lecturas_en_rango`.

3.  **`ResistividadSensor` (Subclase de `Sensor`)**
    *   **Rol:** Sensor especializado en mediciones de resistividad.
    *   **Atributos:** Hereda de `Sensor`.
    *   **Invariantes:**
        *   Acepta unidades como "Ohm·m" o "Ω·m". La `unidad_base` se normalizará a una de ellas o se rechazará.

4.  **`IPSensor` (Subclase de `Sensor`)**
    *   **Rol:** Sensor especializado en mediciones de polarización inducida (IP) / cargabilidad.
    *   **Atributos:** Hereda de `Sensor`.
    *   **Invariantes:**
        *   Acepta unidades como "mV/V".
        *   El método `calcular_indicador_calidad` debe devolver un valor interpretable (e.g., promedio ponderado de valor y calidad de lecturas).

5.  **`Estacion`**
    *   **Rol:** Representa una estación de monitoreo que contiene múltiples sensores.
    *   **Atributos:**
        *   `nombre` (str): Nombre de la estación.
        *   `ubicacion` (tuple): Tupla `(latitud, longitud)` que indica la ubicación de la estación.
        *   `sensores` (dict): Un diccionario donde las claves son los `id` de los sensores y los valores son objetos `Sensor`.
    *   **Invariantes:**
        *   `nombre` no puede ser una cadena vacía.
        *   `ubicacion` debe ser una tupla de dos floats.
        *   No se pueden registrar dos sensores con el mismo `id`.
        *   Solo se pueden registrar objetos de tipo `Sensor` (o sus subclases) en `sensores`.
    *   **Relación:** Composición (una `Estacion` `contiene` `Sensores`).
    *   **Operaciones Clave:** `registrar_sensor`, `obtener_sensor`, `resumen`.


## 2️⃣ 2) Clase `Lectura` (con invariantes y representación)
**Qué construir:** Implementr una clase que **valida** entradas y mantiene invariantes.

**Debe cubrir:**

- Validación mínima de tipos (sin decoradores): comprueba que `x,y,z,valor` sean numéricos; `unidad` sea `str` no vacío; `calidad` ∈ [0,1]; `timestamp` con aspecto ISO ('YYYY-MM-DDTHH:MM:SS').

- Representación legible (texto) para depurar.

- Conversión a dict (para exportar). Opcional: construir desde dict.

**Criterios de aceptación:**

- Si alguna entrada no cumple, el objeto **no debe crearse** (lanza error claro).

- La representación incluye campos relevantes (ts, xyz, valor, unidad, calidad).

**Pistas:**

- Definir un **contrato** corto: rol, invariantes, ejemplo de uso.

- Escribir **2-3 pruebas** (manuales o con `assert`) que fallen se incumplen las reglas (casos borde: calidad=-0.1; unidad="").



In [2]:
import datetime

class Lectura:
    """
    Representa una medición individual realizada por un sensor.

    Atributos:
        timestamp (str): Fecha y hora de la lectura en formato ISO 8601 (YYYY-MM-DDTHH:MM:SS).
        x, y, z (float): Coordenadas espaciales (en metros).
        valor (float): El valor medido por el sensor.
        unidad (str): La unidad de medida del valor.
        calidad (float): Un indicador de calidad de la lectura, entre 0.0 y 1.0.

    Invariantes:
        - timestamp debe ser un string en formato ISO 8601 (YYYY-MM-DDTHH:MM:SS).
        - x, y, z, valor deben ser numéricos (float o int).
        - unidad no puede ser una cadena vacía.
        - calidad debe estar en el rango [0.0, 1.0].
    """

    def __init__(self, timestamp, x, y, z, valor, unidad, calidad):
        # 1. Validar timestamp
        try:
            datetime.datetime.strptime(timestamp, "%Y-%m-%dT%H:%M:%S")
            self.timestamp = timestamp
        except ValueError:
            raise ValueError("Timestamp inválido. Debe ser en formato YYYY-MM-DDTHH:MM:SS.")

        # 2. Validar coordenadas y valor
        if not all(isinstance(coord, (float, int)) for coord in [x, y, z, valor]):
            raise ValueError("x, y, z y valor deben ser numéricos (float o int).")
        self.x = float(x)
        self.y = float(y)
        self.z = float(z)
        self.valor = float(valor)

        # 3. Validar unidad
        if not isinstance(unidad, str) or not unidad:
            raise ValueError("La unidad debe ser una cadena de texto no vacía.")
        self.unidad = unidad

        # 4. Validar calidad
        if not isinstance(calidad, (float, int)) or not (0.0 <= calidad <= 1.0):
            raise ValueError("La calidad debe ser un número entre 0.0 y 1.0.")
        self.calidad = float(calidad)

    def __repr__(self):
        """Representación legible para depuración."""
        return (f"Lectura(TS='{self.timestamp}', XYZ=({self.x:.1f},{self.y:.1f},{self.z:.1f}), "
                f"Valor={self.valor:.2f}{self.unidad}, Calidad={self.calidad:.2f})")

    def to_dict(self):
        """Convierte la lectura a un diccionario para exportación."""
        return {
            "timestamp": self.timestamp,
            "x": self.x,
            "y": self.y,
            "z": self.z,
            "valor": self.valor,
            "unidad": self.unidad,
            "calidad": self.calidad
        }

print("--- Iniciando pruebas de Lectura ---")

# Caso Válido:
print("Probando caso válido...")
try:
    lectura_valida = Lectura("2026-04-30T10:30:00", 10.5, 20.3, 5.0, 100.25, "Ohm·m", 0.95)
    assert lectura_valida.timestamp == "2026-04-30T10:30:00"
    assert lectura_valida.x == 10.5
    assert lectura_valida.valor == 100.25
    assert lectura_valida.unidad == "Ohm·m"
    assert lectura_valida.calidad == 0.95
    print("Lectura válida creada exitosamente.")
except ValueError as e:
    print(f"Falló la creación de lectura válida: {e}")

# Caso Borde: Calidad fuera de rango (menor a 0)
print("\nProbando calidad < 0...")
try:
    Lectura("2026-04-30T10:31:00", 11.0, 21.0, 5.1, 101.0, "Ohm·m", -0.1)
    print("Error: Se creó una lectura con calidad inválida.")
except ValueError as e:
    assert "entre 0.0 y 1.0" in str(e)
    print(f"Error esperado al crear lectura con calidad < 0: {e}")

# Caso Borde: Calidad fuera de rango (mayor a 1)
print("\nProbando calidad > 1...")
try:
    Lectura("2026-04-30T10:32:00", 12.0, 22.0, 5.2, 102.0, "Ohm·m", 1.1)
    print("Error: Se creó una lectura con calidad inválida.")
except ValueError as e:
    assert "entre 0.0 y 1.0" in str(e)
    print(f"Error esperado al crear lectura con calidad > 1: {e}")

# Caso Error: Unidad vacía
print("\nProbando unidad vacía...")
try:
    Lectura("2026-04-30T10:33:00", 13.0, 23.0, 5.3, 103.0, "", 0.7)
    print("Error: Se creó una lectura con unidad vacía.")
except ValueError as e:
    assert "no vacía" in str(e)
    print(f"Error esperado al crear lectura con unidad vacía: {e}")

# Caso Error: Timestamp mal formado
print("\nProbando timestamp mal formado...")
try:
    Lectura("2026-04-30 10:34:00", 14.0, 24.0, 5.4, 104.0, "Ohm·m", 0.8)
    print("Error: Se creó una lectura con timestamp mal formado.")
except ValueError as e:
    assert "Timestamp inválido" in str(e)
    print(f"Error esperado al crear lectura con timestamp mal formado: {e}")

# Prueba de to_dict
print("\nProbando método to_dict...")
dict_lectura = lectura_valida.to_dict()
assert isinstance(dict_lectura, dict)
assert dict_lectura["timestamp"] == "2026-04-30T10:30:00"
assert dict_lectura["valor"] == 100.25
print("Método to_dict funciona correctamente.")

print("--- Pruebas de Lectura finalizadas ---")


--- Iniciando pruebas de Lectura ---
Probando caso válido...
Lectura válida creada exitosamente.

Probando calidad < 0...
Error esperado al crear lectura con calidad < 0: La calidad debe ser un número entre 0.0 y 1.0.

Probando calidad > 1...
Error esperado al crear lectura con calidad > 1: La calidad debe ser un número entre 0.0 y 1.0.

Probando unidad vacía...
Error esperado al crear lectura con unidad vacía: La unidad debe ser una cadena de texto no vacía.

Probando timestamp mal formado...
Error esperado al crear lectura con timestamp mal formado: Timestamp inválido. Debe ser en formato YYYY-MM-DDTHH:MM:SS.

Probando método to_dict...
Método to_dict funciona correctamente.
--- Pruebas de Lectura finalizadas ---



## 3️⃣ 3) `Sensor` (base): encapsulamiento y operaciones
**Qué construir:** Clase base con atributos mínimos y una colección interna de lecturas.

**Debe cubrir:**

- Atributos: `id`, `tipo`, `unidad_base`, `lecturas` (lista interna).

- Operaciones: agregar lectura; contar lecturas; obtener última; calcular promedio (considerando un **umbral de calidad** parametrizable); devolver valores en un **rango temporal**.

- Verificar **compatibilidad de unidad**: en la base puedes exigir coincidencia exacta.

**Criterios de aceptación:**

- No se permite agregar algo que **no** sea `Lectura`.

- El promedio retorna `None` si no hay datos válidos.

- Un método de **búsqueda por rango temporal** funciona con strings ISO (orden lexicográfico simple).

**Pistas:**

- Decidir qué métodos se exponen públicamente y cuáles se consideran “internos” (por convención de nombres).

- Agregar al menos **3 pruebas**: agregar lecturas válidas/ inválidas; promedio con y sin umbral; última lectura.



In [3]:
class Sensor:
    """
    Clase base que representa un sensor genérico.
    Encapsula una lista de lecturas y provee operaciones fundamentales.
    """
    def __init__(self, id_sensor, tipo, unidad_base):
        if not id_sensor or not tipo or not unidad_base:
            raise ValueError("ID, tipo y unidad_base son campos obligatorios.")

        self.id = id_sensor
        self.tipo = tipo
        self.unidad_base = unidad_base
        self._lecturas = []  # Atributo 'protegido' para encapsular los datos

    def __repr__(self):
        return f"Sensor({self.id}, {self.tipo}, {len(self._lecturas)} lecturas)"

    def aceptar_unidad(self, unidad):
        """
        Verifica si la unidad de una lectura es compatible con el sensor.
        En la clase base, se requiere coincidencia exacta.
        """
        return unidad == self.unidad_base

    def agregar_lectura(self, lectura):
        """
        Agrega una instancia de Lectura validando tipo y unidad.
        """
        if not isinstance(lectura, Lectura):
            raise TypeError("El objeto debe ser una instancia de la clase Lectura.")

        if not self.aceptar_unidad(lectura.unidad):
            raise ValueError(f"Unidad '{lectura.unidad}' no compatible con '{self.unidad_base}'.")

        self._lecturas.append(lectura)

    def contar_lecturas(self):
        return len(self._lecturas)

    def obtener_ultima_lectura(self):
        return self._lecturas[-1] if self._lecturas else None

    def calcular_promedio(self, umbral_calidad=0.0):
        """
        Calcula el promedio de los valores de las lecturas que cumplen
        con el criterio de calidad mínimo.
        """
        lecturas_validas = [l.valor for l in self._lecturas if l.calidad >= umbral_calidad]

        if not lecturas_validas:
            return None

        return sum(lecturas_validas) / len(lecturas_validas)

    def filtrar_por_rango(self, inicio_iso, fin_iso):
        """
        Retorna una lista de lecturas dentro de un rango de tiempo ISO.
        """
        return [l for l in self._lecturas if inicio_iso <= l.timestamp <= fin_iso]

print("Probando clase Sensor...")
s_test = Sensor("S-01", "Genérico", "m")

l1 = Lectura("2026-05-01T10:00:00", 0, 0, 0, 50.0, "m", 0.9)
l2 = Lectura("2026-05-01T11:00:00", 0, 0, 0, 60.0, "m", 0.4)

s_test.agregar_lectura(l1)
s_test.agregar_lectura(l2)

print(f"Total lecturas: {s_test.contar_lecturas()}")
print(f"Promedio (calidad > 0.5): {s_test.calcular_promedio(0.5)}")
print(f"Promedio (todas): {s_test.calcular_promedio(0.0)}")

try:
    l_error = Lectura("2026-05-01T12:00:00", 0, 0, 0, 10.0, "km", 0.9)
    s_test.agregar_lectura(l_error)
except ValueError as e:
    print(f"Error de unidad capturado: {e}")

Probando clase Sensor...
Total lecturas: 2
Promedio (calidad > 0.5): 50.0
Promedio (todas): 55.0
Error de unidad capturado: Unidad 'km' no compatible con 'm'.



## 4️⃣ 4) Herencia y Polimorfismo (`ResistividadSensor`, `IPSensor`)
**Qué construir:** Dos subclases con reglas específicas y al menos **un método propio**.

**Debe cubrir:**

- `ResistividadSensor`: considerar **equivalentes** `Ohm·m` y `Ω·m`.

- `IPSensor`: acepta `mV/V` y ofrece un **método de calidad** (defínelo tú) que combine alguna medida del valor con la calidad de las lecturas.

- Demostrar **polimorfismo**: un mismo “mensaje” (p. ej., `promedio()`) aplicado a ambas subclases.

**Criterios de aceptación:**

- La validación de unidad **difiere** entre la base y las subclases.

- El método propio de `IPSensor` produce un **indicador interpretable** (describe cómo leerlo).

**Pistas:**

- Describir supuestos (no físicos) para el indicador: cómo se normaliza, cómo se pondera.

- Agregar pruebas: que `ResistividadSensor` acepte `Ω·m` y rechace otras; que el indicador de `IPSensor` cambie al variar datos.



In [4]:
class ResistividadSensor(Sensor):
    """
    Subclase especializada en resistividad.
    Acepta tanto 'Ohm·m' como 'Ω·m' como unidades válidas.
    """
    def __init__(self, id_sensor, unidad_base="Ohm·m"):
        super().__init__(id_sensor, "Resistividad DC", unidad_base)

    def aceptar_unidad(self, unidad):
        # Polimorfismo: Regla específica para resistividad
        equivalentes = {"Ohm·m", "Ω·m"}
        return unidad in equivalentes and self.unidad_base in equivalentes

class IPSensor(Sensor):
    """
    Subclase especializada en Polarización Inducida.
    Incluye un método propio para calcular un indicador de calidad avanzado.
    """
    def __init__(self, id_sensor, unidad_base="mV/V"):
        super().__init__(id_sensor, "IP/Chargeability", unidad_base)

    def calcular_indicador_calidad_ip(self):
        """
        Método propio: Retorna un indicador (0-100) basado en el promedio
        de calidad ponderado por la estabilidad de las lecturas.
        """
        if not self._lecturas:
            return 0.0

        promedio_calidad = self.calcular_promedio_calidad()
        # Simulación de indicador: calidad base * 100
        return promedio_calidad * 100

    def calcular_promedio_calidad(self):
        if not self._lecturas: return 0.0
        return sum(l.calidad for l in self._lecturas) / len(self._lecturas)

# --- Pruebas Sección 4 ---
print("--- Probando Herencia y Polimorfismo ---")

# Probar Resistividad (Equivalencia de unidades)
res_sensor = ResistividadSensor("RES-99")
l_ohm = Lectura("2026-05-02T09:00:00", 0,0,0, 400.0, "Ohm·m", 0.9)
l_omega = Lectura("2026-05-02T09:10:00", 0,0,0, 405.0, "Ω·m", 0.8)

try:
    res_sensor.agregar_lectura(l_ohm)
    res_sensor.agregar_lectura(l_omega)
    print(f"✅ Resistividad aceptó ambas unidades (Ohm·m y Ω·m).")
except ValueError as e:
    print(f"❌ Error en unidades de resistividad: {e}")

# Probar IP e indicador propio
ip_sensor = IPSensor("IP-55")
ip_sensor.agregar_lectura(Lectura("2026-05-02T10:00:00", 0,0,0, 12.5, "mV/V", 0.95))
ip_sensor.agregar_lectura(Lectura("2026-05-02T10:05:00", 0,0,0, 11.8, "mV/V", 0.85))

indicador = ip_sensor.calcular_indicador_calidad_ip()
print(f"✅ Indicador Calidad IP para {ip_sensor.id}: {indicador:.1f}/100")

# Demostración de Polimorfismo
sensores = [res_sensor, ip_sensor]
print("\nResumen polimórfico (Promedios):")
for s in sensores:
    print(f"- {s.tipo} ({s.id}): Promedio = {s.calcular_promedio():.2f} {s.unidad_base}")

--- Probando Herencia y Polimorfismo ---
✅ Resistividad aceptó ambas unidades (Ohm·m y Ω·m).
✅ Indicador Calidad IP para IP-55: 90.0/100

Resumen polimórfico (Promedios):
- Resistividad DC (RES-99): Promedio = 402.50 Ohm·m
- IP/Chargeability (IP-55): Promedio = 12.15 mV/V



## 5️⃣ 5) `Estacion`: composición, búsqueda y resumen
**Qué construir:** Clase que gestiona múltiples sensores, con métodos de **registro**, **acceso** y **resumen**.

**Debe cubrir:**

- Registrar sensor (evita duplicados por `id`).

- Obtener sensor por `id`.

- `resumen()`: estructura con totales por sensor (tipo, unidad, nº lecturas, promedio y —si aplica— indicador de calidad IP).

**Criterios de aceptación:**

- Registrar el mismo `id` dos veces debe **fallar** con un mensaje claro.

- `resumen()` devuelve un objeto (dict o similar) fácil de serializar.

**Pistas:**

- Pensar en cómo presentar el resumen para que luego sea **exportable** sin fricción.

- Escribir una prueba que cree una estación, registrar 2 sensores y confirmar los conteos/promedios.



In [5]:
class Estacion:
    """
    Gestiona múltiples sensores en una ubicación geográfica.
    Aplica composición: una Estación tiene N Sensores.
    """
    def __init__(self, nombre, ubicacion):
        if not nombre:
            raise ValueError("El nombre de la estación es obligatorio.")
        if not isinstance(ubicacion, tuple) or len(ubicacion) != 2:
            raise ValueError("La ubicación debe ser una tupla (lat, lon).")

        self.nombre = nombre
        self.ubicacion = ubicacion
        self.sensores = {}  # Diccionario id -> Sensor

    def registrar_sensor(self, sensor):
        if not isinstance(sensor, Sensor):
            raise TypeError("Solo se pueden registrar objetos de tipo Sensor o sus subclases.")

        if sensor.id in self.sensores:
            raise ValueError(f"Error: El sensor con ID '{sensor.id}' ya está registrado en esta estación.")

        self.sensores[sensor.id] = sensor

    def obtener_sensor(self, id_sensor):
        return self.sensores.get(id_sensor)

    def resumen(self):
        """
        Genera un diccionario con el estado actual de todos los sensores.
        """
        informe = {
            "estacion": self.nombre,
            "ubicacion": self.ubicacion,
            "total_sensores": len(self.sensores),
            "detalles": []
        }

        for id_s, s in self.sensores.items():
            datos = {
                "id": id_s,
                "tipo": s.tipo,
                "unidad": s.unidad_base,
                "n_lecturas": s.contar_lecturas(),
                "promedio": s.calcular_promedio()
            }

            # Si es un IPSensor, agregamos su indicador especial
            if isinstance(s, IPSensor):
                datos["indicador_calidad_ip"] = s.calcular_indicador_calidad_ip()

            informe["detalles"].append(datos)

        return informe

print("--- Probando Clase Estacion ---")

estacion_norte = Estacion("Estación Norte", (-23.5, -70.1))

s_res = ResistividadSensor("RES-001")
s_ip = IPSensor("IP-001")

# Registro exitoso
estacion_norte.registrar_sensor(s_res)
estacion_norte.registrar_sensor(s_ip)
print(f"✅ Sensores registrados en {estacion_norte.nombre}.")

# Prueba de duplicado
try:
    estacion_norte.registrar_sensor(ResistividadSensor("RES-001"))
except ValueError as e:
    print(f"✅ Capturado error de duplicado: {e}")

import json
print("\nResumen de la estación:")
print(json.dumps(estacion_norte.resumen(), indent=2, ensure_ascii=False))

--- Probando Clase Estacion ---
✅ Sensores registrados en Estación Norte.
✅ Capturado error de duplicado: Error: El sensor con ID 'RES-001' ya está registrado en esta estación.

Resumen de la estación:
{
  "estacion": "Estación Norte",
  "ubicacion": [
    -23.5,
    -70.1
  ],
  "total_sensores": 2,
  "detalles": [
    {
      "id": "RES-001",
      "tipo": "Resistividad DC",
      "unidad": "Ohm·m",
      "n_lecturas": 0,
      "promedio": null
    },
    {
      "id": "IP-001",
      "tipo": "IP/Chargeability",
      "unidad": "mV/V",
      "n_lecturas": 0,
      "promedio": null,
      "indicador_calidad_ip": 0.0
    }
  ]
}



## 6️⃣ 6) Manejo de errores (experiencia de usuario)
**Qué construir:** Un patrón de uso “seguro” para registrar sensores/lecturas y **reportar errores** de forma entendible.

**Debe cubrir:**

- Mensajes breves (qué pasó y cómo corregirlo).

- Ejemplos de errores: duplicado de `id`, unidad incompatible, lectura mal formada.

**Criterios de aceptación:**

- Al menos **3 casos** de error documentados con la salida esperada.

**Pistas:**

- Evitar mensajes crípticos; sugiere **acción correctiva** (“Verifica la unidad esperada: mV/V”).



In [6]:
def demostrar_errores():
    print("--- Simulando Errores de Usuario ---")

    estacion_u = Estacion("Estación Central", (-33.4, -70.6))
    s_res = ResistividadSensor("RES-500")
    estacion_u.registrar_sensor(s_res)

    # Caso 1: Error de ID Duplicado
    print("\n[Caso 1: Registro Duplicado]")
    try:
        s_duplicado = ResistividadSensor("RES-500")
        estacion_u.registrar_sensor(s_duplicado)
    except ValueError as e:
        print(f"Mensaje del sistema: {e}")
        print("Acción correctiva: Verifica que el ID sea único para esta estación.")

    # Caso 2: Error de Unidad Incompatible
    print("\n[Caso 2: Unidad Incompatible]")
    try:
        # ResistividadSensor espera Ohm·m o Ω·m
        l_mala_unidad = Lectura("2026-05-01T12:00:00", 0, 0, 0, 100.0, "Celsius", 0.9)
        s_res.agregar_lectura(l_mala_unidad)
    except ValueError as e:
        print(f"Mensaje del sistema: {e}")
        print(f"Acción correctiva: Verifica la unidad esperada para {s_res.tipo}.")

    # Caso 3: Error de Calidad fuera de rango
    print("\n[Caso 3: Calidad Inválida]")
    try:
        Lectura("2026-05-01T12:00:00", 10, 10, 10, 50.0, "Ohm·m", 1.5)
    except ValueError as e:
        print(f"Mensaje del sistema: {e}")
        print("Acción correctiva: El valor de calidad debe ser un decimal entre 0.0 y 1.0.")

    # Caso 4: Registro de objeto no válido
    print("\n[Caso 4: Tipo de Objeto Incorrecto]")
    try:
        estacion_u.registrar_sensor("Un String no es un Sensor")
    except TypeError as e:
        print(f"Mensaje del sistema: {e}")

demostrar_errores()

--- Simulando Errores de Usuario ---

[Caso 1: Registro Duplicado]
Mensaje del sistema: Error: El sensor con ID 'RES-500' ya está registrado en esta estación.
Acción correctiva: Verifica que el ID sea único para esta estación.

[Caso 2: Unidad Incompatible]
Mensaje del sistema: Unidad 'Celsius' no compatible con 'Ohm·m'.
Acción correctiva: Verifica la unidad esperada para Resistividad DC.

[Caso 3: Calidad Inválida]
Mensaje del sistema: La calidad debe ser un número entre 0.0 y 1.0.
Acción correctiva: El valor de calidad debe ser un decimal entre 0.0 y 1.0.

[Caso 4: Tipo de Objeto Incorrecto]
Mensaje del sistema: Solo se pueden registrar objetos de tipo Sensor o sus subclases.



## 7️⃣ 7) Persistencia: exportar resultados (JSON/CSV)
**Qué construir:** Utilidades o métodos que exporten lecturas de un sensor y/o el resumen de una estación a **JSON** y **CSV**.

**Debe cubrir:**

- Definir **nombres de campos** consistentes (`timestamp,x,y,z,valor,unidad,calidad`).

- Asegurar que la carpeta de salida exista (o documentar el requisito).

**Criterios de aceptación:**

- Se generan **dos archivos** mínimos (uno JSON y uno CSV) a partir de tus clases.

- La exportación respeta el orden/nombres de columnas.

**Pistas:**

- Describir cómo **testear** la exportación (abrir el archivo y revisar campos/filas). No incluir código aquí, sólo el plan.



In [7]:
import json
import csv
import os

def exportar_a_json(estacion, filename="estacion_data.json"):
    """Exporta el resumen de la estación a un archivo JSON."""
    data = estacion.resumen()
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    print(f"✅ Datos exportados exitosamente a {filename}")

def exportar_lecturas_csv(sensor, filename="lecturas_sensor.csv"):
    """Exporta las lecturas de un sensor específico a CSV."""
    # Campos requeridos
    campos = ['timestamp', 'x', 'y', 'z', 'valor', 'unidad', 'calidad']

    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=campos)
        writer.writeheader()
        for lectura in sensor._lecturas:
            writer.writerow(lectura.to_dict())
    print(f"✅ Lecturas del sensor {sensor.id} exportadas a {filename}")

# Aseguramos que existan datos para exportar
if 'estacion_norte' in locals():
    # Exportar resumen completo
    exportar_a_json(estacion_norte, "resumen_estacion_norte.json")

    # Exportar lecturas del sensor de resistividad
    sensor_r = estacion_norte.obtener_sensor("RES-001")
    if sensor_r:
        exportar_lecturas_csv(sensor_r, "lecturas_res_001.csv")

✅ Datos exportados exitosamente a resumen_estacion_norte.json
✅ Lecturas del sensor RES-001 exportadas a lecturas_res_001.csv



## 8️⃣ 8) Pruebas y mini-benchmark (ligero)
**Qué construir:** Un conjunto breve de **pruebas** y una **medición** simple que compare acceso a sensores por `id`.

**Debe cubrir:**

- Pruebas mínimas por sección (2–3 cada una) y un checklist de resultados esperados.

- Mini-benchmark: medición del tiempo de **acceso por diccionario** a un sensor repetidas veces, y un comentario de lo observado.

**Criterios de aceptación:**

- Documentas cómo correr las pruebas y qué salida esperar.

- Incluyes una breve **interpretación** del benchmark (en una frase).

**Pistas:**

- No buscar microsegundos exactos; enfatizar el **orden de magnitud** y la razón del diccionario.



In [8]:
import time

def ejecutar_pruebas():
    print("--- Iniciando Suite de Pruebas (Sección 8) ---")

    # 1. Prueba de Integridad de Estación
    e = Estacion("Test Lab", (0.0, 0.0))
    s1 = ResistividadSensor("R1")
    e.registrar_sensor(s1)
    assert e.obtener_sensor("R1") == s1, "Error: No se recuperó el sensor correcto"
    print("✅ Prueba 1: Registro y recuperación de sensores exitosa.")

    # 2. Prueba de Promedio con Umbral de Calidad
    s1.agregar_lectura(Lectura("2026-01-01T10:00:00", 0,0,0, 100, "Ohm·m", 0.9))
    s1.agregar_lectura(Lectura("2026-01-01T10:05:00", 0,0,0, 50, "Ohm·m", 0.1)) # Calidad baja

    prom_alto = s1.calcular_promedio(umbral_calidad=0.8)
    prom_todo = s1.calcular_promedio(umbral_calidad=0.0)

    assert prom_alto == 100, f"Error en promedio filtrado: {prom_alto}"
    assert prom_todo == 75, f"Error en promedio total: {prom_todo}"
    print("✅ Prueba 2: Cálculo de promedios con umbral de calidad correcto.")

    # 3. Prueba de Polimorfismo (IP Indicator)
    s_ip = IPSensor("IP1")
    s_ip.agregar_lectura(Lectura("2026-01-01T11:00:00", 0,0,0, 10, "mV/V", 1.0))
    assert s_ip.calcular_indicador_calidad_ip() == 100.0, "Error en indicador IP"
    print("✅ Prueba 3: Polimorfismo e indicadores específicos de IPSensor correctos.")

def mini_benchmark():
    print("\n--- Mini-Benchmark: Acceso por Diccionario ---")
    e_bench = Estacion("Bench", (0,0))
    # Crear 1000 sensores ficticios
    for i in range(1000):
        e_bench.registrar_sensor(Sensor(f"S-{i}", "Test", "u"))

    id_buscar = "S-999"
    repeticiones = 100000

    inicio = time.time()
    for _ in range(repeticiones):
        _ = e_bench.obtener_sensor(id_buscar)
    fin = time.time()

    tiempo_total = fin - inicio
    print(f"Resultados: {repeticiones} accesos realizados en {tiempo_total:.4f} segundos.")
    print("Interpretación: El acceso mediante diccionario es O(1), lo que permite recuperar sensores casi instantáneamente sin importar cuántos haya registrados.")

ejecutar_pruebas()
mini_benchmark()

--- Iniciando Suite de Pruebas (Sección 8) ---
✅ Prueba 1: Registro y recuperación de sensores exitosa.
✅ Prueba 2: Cálculo de promedios con umbral de calidad correcto.
✅ Prueba 3: Polimorfismo e indicadores específicos de IPSensor correctos.

--- Mini-Benchmark: Acceso por Diccionario ---
Resultados: 100000 accesos realizados en 0.0168 segundos.
Interpretación: El acceso mediante diccionario es O(1), lo que permite recuperar sensores casi instantáneamente sin importar cuántos haya registrados.



## 9️⃣ 9) Desafíos (opcionales)
- Diseñar `GPSensor` (unidad `m` o `deg`) con un método `distancia_total()` (describe algoritmo: suma de segmentos sucesivos).

- Agregar **deduplicación** de `Lectura` por `(timestamp, x, y, z, unidad)` (explica criterio).

- Implementar un **importador** desde JSON/CSV a tus clases (detalla mapeo de campos y validaciones).

- Comparar tu diseño original con una variante **sin herencia** (usa composición) y comenta pros/contras.




## 🔟 10) Checklist de entrega
- [ ] Diagrama textual y descripción de clases con **invariantes**.

- [ ] `Lectura` válida frente a casos borde.

- [ ] `Sensor` con operaciones clave y umbral de calidad.

- [ ] Subclases con reglas de unidad y método propio (IP).

- [ ] `Estacion` con registro seguro, acceso por `id` y `resumen()` exportable.

- [ ] Manejo de errores con mensajes claros.

- [ ] Exportación JSON/CSV funcionando.

- [ ] Pruebas mínimas y mini-benchmark documentados.

- [ ] Comentario de diseño (2–3 decisiones justificadas).




## ✍️ Guía de evaluación
- **Modelado e invariantes (20%)**: claridad del dominio, invariantes bien definidos.

- **Diseño de clases (25%)**: encapsulamiento, operaciones y relaciones coherentes.

- **Herencia/polimorfismo (15%)**: subclases con reglas significativas y demostración.

- **Calidad de validaciones/errores (10%)**: mensajes útiles, cobertura de casos borde.

- **Persistencia (10%)**: exportación correcta y consistente.

- **Pruebas/benchmark (10%)**: suficiencia y análisis breve.

- **Comunicación (10%)**: documentación, decisiones y orden del cuaderno.



## Ejemplo:

---



```
# ============================================
# Ejemplo de clase y subclases en Python
# Contexto: sistema de salas de una universidad
# ============================================

class Sala:
    """
    Clase base que representa una sala general de la universidad.
    """

    def __init__(self, codigo, capacidad, edificio):
        self.codigo = codigo
        self.capacidad = capacidad
        self.edificio = edificio
        self.disponible = True

    def mostrar_informacion(self):
        print("Información de la sala")
        print(f"Código: {self.codigo}")
        print(f"Capacidad: {self.capacidad} personas")
        print(f"Edificio: {self.edificio}")
        print(f"Disponible: {'Sí' if self.disponible else 'No'}")

    def reservar(self):
        if self.disponible:
            self.disponible = False
            print(f"La sala {self.codigo} fue reservada correctamente.")
        else:
            print(f"La sala {self.codigo} no está disponible.")

    def liberar(self):
        self.disponible = True
        print(f"La sala {self.codigo} fue liberada.")


# Subclase 1: hereda desde Sala
class SalaComputacion(Sala):
    """
    Subclase que representa una sala de computación.
    Hereda atributos y métodos de Sala, pero agrega nuevos elementos.
    """

    def __init__(self, codigo, capacidad, edificio, cantidad_computadores, tiene_proyector):
        super().__init__(codigo, capacidad, edificio)
        self.cantidad_computadores = cantidad_computadores
        self.tiene_proyector = tiene_proyector

    def mostrar_informacion(self):
        super().mostrar_informacion()
        print(f"Cantidad de computadores: {self.cantidad_computadores}")
        print(f"Tiene proyector: {'Sí' if self.tiene_proyector else 'No'}")

    def verificar_equipamiento(self):
        if self.cantidad_computadores >= self.capacidad:
            print("La sala tiene computadores suficientes para todos los estudiantes.")
        else:
            print("La sala no tiene computadores suficientes para todos los estudiantes.")


# Subclase 2: hereda desde Sala
class SalaLaboratorio(Sala):
    """
    Subclase que representa un laboratorio.
    Hereda desde Sala y agrega atributos propios.
    """

    def __init__(self, codigo, capacidad, edificio, tipo_laboratorio, requiere_supervision):
        super().__init__(codigo, capacidad, edificio)
        self.tipo_laboratorio = tipo_laboratorio
        self.requiere_supervision = requiere_supervision

    def mostrar_informacion(self):
        super().mostrar_informacion()
        print(f"Tipo de laboratorio: {self.tipo_laboratorio}")
        print(f"Requiere supervisión: {'Sí' if self.requiere_supervision else 'No'}")

    def indicar_condiciones_uso(self):
        if self.requiere_supervision:
            print("Este laboratorio solo puede usarse con supervisión docente.")
        else:
            print("Este laboratorio puede usarse sin supervisión especial.")
```

